In [ ]:
"""
Standalone DBSCANR implementation translated from MATLAB.

Original MATLAB function signature:
    U = DBSCANR(Data, k, K, p, expand, enforceK)

Notes:
- Labels follow MATLAB style: 0 = unlabeled/noise, clusters start from 1.
- Distance in MinkDist is sum(abs(x-y)^p), i.e. WITHOUT taking 1/p root,
  matching the MATLAB code.
"""

import numpy as np


class DBSCANR:
    def __init__(self, k, K=np.inf, p=2, expand=True, enforceK=False):
        self.k = int(k)
        self.K = K
        self.p = p
        self.expand = bool(expand)
        self.enforceK = bool(enforceK)

    def fit_predict(self, data):
        data = np.asarray(data, dtype=float)
        M = data.shape[0]

        U = np.zeros(M, dtype=int)  # MATLAB: U(1:M,1)=0
        C = 1

        Q, Q_flag, p_dist = self._rnn(data, self.k, self.p)

        if not Q_flag:
            core_pts_mat, core_pts_idx, _ = self._get_core_pts_mat(Q, M, self.k, order=(True, -2))
            ranked_core_pts_idxs = core_pts_mat[:, 0].astype(int) if core_pts_mat.size > 0 else np.array([], dtype=int)

            if ranked_core_pts_idxs.size > 0 and ranked_core_pts_idxs.size >= 2:
                while ranked_core_pts_idxs.size > 0:
                    q_idx = int(ranked_core_pts_idxs[0])

                    # MATLAB: setdiff(..., qIdx, 'stable')
                    ranked_core_pts_idxs = self._setdiff_stable(ranked_core_pts_idxs, np.array([q_idx], dtype=int))

                    neighbours, _ = self._search_neighbourhood(q_idx, Q, core_pts_mat[:, 0].astype(int))

                    U, S = self._expand_cluster(q_idx, neighbours, C, U, Q, core_pts_mat[:, 0].astype(int))
                    if S >= self.k:
                        C = C + 1

                    if C > self.K and self.enforceK:
                        ranked_core_pts_idxs = np.array([], dtype=int)

                    # MATLAB: setdiff(RankedCorePtsIdxs, find(U>0), 'stable')
                    ranked_core_pts_idxs = self._setdiff_stable(ranked_core_pts_idxs, np.where(U > 0)[0].astype(int))

            if self.expand:
                U = self._get_clustering(p_dist, U, M, core_pts_mat[:, 0].astype(int) if core_pts_mat.size > 0 else np.array([], dtype=int))

        return U

    @staticmethod
    def _setdiff_stable(a, b):
        bset = set(np.asarray(b, dtype=int).tolist())
        return np.array([x for x in np.asarray(a, dtype=int).tolist() if x not in bset], dtype=int)

    def _mink_dist(self, data, k, p, M):
        dist = np.zeros((M, M), dtype=float)
        kdist = np.full((M,), np.nan, dtype=float)

        for i in range(M):
            # MATLAB: sum((abs(repmat(Data(i,:),M,1)-Data).^p),2);
            diff = np.abs(data[i, :] - data) ** p
            dist[i, :] = np.sum(diff, axis=1)

            tmp = np.sort(dist[i, :])
            # MATLAB kdist(i,1)=tmp(k+1,1) due to self-distance at position 1 (1-based)
            kdist[i] = tmp[k]  # 0-based equivalent

        return dist, kdist

    def _rnn(self, data, k, p):
        M = data.shape[0]

        if k >= M:
            R = np.zeros((M, M), dtype=bool)
            p_dist = np.zeros((M, M), dtype=float)
            return R, True, p_dist

        p_dist, kdist = self._mink_dist(data, k, p, M)

        # MATLAB: kdist = kdist(:,IdxOnes) -> repeat by columns
        kdist_mat = np.repeat(kdist.reshape(M, 1), M, axis=1)

        # MATLAB: R = p_dist <= kdist
        R = p_dist <= kdist_mat

        # MATLAB: R = R - eye(M)
        np.fill_diagonal(R, False)

        return R, False, p_dist

    def _get_core_pts_mat(self, Q, M, k, order=(True, -2)):
        pts_idx = np.arange(M, dtype=int)
        rnn_mat = np.sum(Q, axis=0)  # MATLAB sum(Q,1)
        core_pts_idx = pts_idx[rnn_mat >= k]
        nRNN = rnn_mat[core_pts_idx]

        if core_pts_idx.size == 0:
            return np.empty((0, 2), dtype=float), core_pts_idx, nRNN

        core_pts_mat = np.column_stack([core_pts_idx, nRNN])

        if order[0]:
            j = order[1]
            if j == -2:
                # sortrows by 2nd col descending
                idx = np.argsort(-core_pts_mat[:, 1], kind="mergesort")
                core_pts_mat = core_pts_mat[idx, :]

        return core_pts_mat, core_pts_idx, nRNN

    def _search_neighbourhood(self, q_idx, Q, core_pts):
        # MATLAB:
        # Neighbours = find(Q(:,qIdx))';
        # Neighbours = CorePts(ismember(CorePts',Neighbours));
        neighbours = np.where(Q[:, q_idx])[0].astype(int)
        neighbours = np.array([c for c in core_pts if c in set(neighbours.tolist())], dtype=int)

        # MATLAB:
        # NearestNeighbours = find(Q(qIdx,:))';
        nearest_neighbours = np.where(Q[q_idx, :])[0].astype(int)

        return neighbours, nearest_neighbours

    def _expand_cluster(self, q_idx, neighbours, C, U, Q, core_pts):
        oldU = U.copy()
        U[q_idx] = C

        neighbours = np.array(neighbours, dtype=int).copy()
        idx = 0

        while neighbours.size > 0:
            # MATLAB pops from front:
            j = int(neighbours[idx])
            neighbours = np.delete(neighbours, idx)

            if U[j] == 0:
                search_n, _ = self._search_neighbourhood(j, Q, core_pts)

                # MATLAB: [Neighbours; setdiff(..., Neighbours)]
                to_add = self._setdiff_stable(search_n, neighbours)
                if to_add.size > 0:
                    neighbours = np.concatenate([neighbours, to_add])

                # MATLAB: Neighbours = setdiff(Neighbours, Neighbours(U(Neighbours)>0));
                if neighbours.size > 0:
                    assigned = neighbours[U[neighbours] > 0]
                    neighbours = self._setdiff_stable(neighbours, assigned)

                U[j] = C

        # MATLAB:
        # currentClusterIdx = find(U-oldU);
        current_cluster_idx = np.where(U - oldU != 0)[0]

        for i in current_cluster_idx:
            _, nearest_neighbours = self._search_neighbourhood(int(i), Q, core_pts)
            nearest_neighbours = self._setdiff_stable(nearest_neighbours, core_pts)
            if nearest_neighbours.size > 0:
                U[nearest_neighbours] = C

        S = int(np.sum(oldU != U))
        return U, S

    def _get_clustering(self, p_dist, U, M, core_pts):
        no_of_obj_idxs = np.arange(M, dtype=int)
        labeled_obj_idxs = no_of_obj_idxs[U != 0]
        core_idxs = np.intersect1d(labeled_obj_idxs, core_pts, assume_unique=False)
        unlabeled_obj_idxs = no_of_obj_idxs[U == 0]
        L = int(unlabeled_obj_idxs.size)

        if unlabeled_obj_idxs.size > 0:
            # MATLAB: if L <= M * 0.5
            if L <= M * 0.5:
                for q_idx in unlabeled_obj_idxs:
                    if core_idxs.size == 0:
                        break
                    # MATLAB:
                    # tmp = [(1:M)', p_dist(:,qIdx)];
                    # sortedCores = sortrows(tmp(CoreIdxs,:),2);
                    d = p_dist[core_idxs, q_idx]
                    best = core_idxs[np.argmin(d)]
                    U[q_idx] = U[best]

        return U